# 04 - Single-Qubit Gates

Explore all single-qubit gates in QuantSDK's gate library.

**Concepts:** Pauli gates, Clifford gates, rotation gates, parameterized gates

In [ ]:
import quantsdk as qs
import math

## Pauli Gates (X, Y, Z)

- **X** (NOT): Flips |0> to |1> and vice versa
- **Y**: Rotation around Y-axis by pi
- **Z**: Phase flip (|1> gets a minus sign)

In [ ]:
# X gate: NOT
circuit = qs.Circuit(1).x(0).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"X|0> = {result.counts}")  # Always |1>

# Y gate
circuit = qs.Circuit(1).y(0).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"Y|0> = {result.counts}")  # Always |1> (with phase)

# Z gate
circuit = qs.Circuit(1).z(0).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"Z|0> = {result.counts}")  # Still |0> (Z only affects phase of |1>)

## Hadamard Gate (H)

Creates equal superposition.

In [ ]:
# Single Hadamard
circuit = qs.Circuit(1).h(0).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"H|0> = {result.counts}")

# Double Hadamard = Identity
circuit = qs.Circuit(1).h(0).h(0).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"H*H|0> = {result.counts}")  # Back to |0>

## Phase Gates (S, Sdg, T, Tdg, SX, SXdg)

In [ ]:
gates = [
    ("S",    lambda c: c.s(0)),
    ("Sdg",  lambda c: c.sdg(0)),
    ("T",    lambda c: c.t(0)),
    ("Tdg",  lambda c: c.tdg(0)),
    ("SX",   lambda c: c.sx(0)),
    ("SXdg", lambda c: c.sxdg(0)),
]

for name, apply_gate in gates:
    circuit = qs.Circuit(1)
    circuit.h(0)     # Put in superposition first
    apply_gate(circuit)
    circuit.h(0)     # Interfere
    circuit.measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    print(f"H-{name:4s}-H|0> = {result.counts}")

## Rotation Gates (RX, RY, RZ)

Parameterized rotations around the Bloch sphere axes.

In [ ]:
# RY sweep: rotate from |0> toward |1>
print("RY rotation sweep:")
for angle_frac in [0, 0.25, 0.5, 0.75, 1.0]:
    theta = angle_frac * math.pi
    circuit = qs.Circuit(1).ry(0, theta).measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    p1 = result.counts.get('1', 0) / 1000
    print(f"  RY({angle_frac:.2f}*pi): P(|1>) = {p1:.3f}")

In [ ]:
# RX sweep
print("RX rotation sweep:")
for angle_frac in [0, 0.25, 0.5, 0.75, 1.0]:
    theta = angle_frac * math.pi
    circuit = qs.Circuit(1).rx(0, theta).measure_all()
    result = qs.run(circuit, shots=1000, seed=42)
    p1 = result.counts.get('1', 0) / 1000
    print(f"  RX({angle_frac:.2f}*pi): P(|1>) = {p1:.3f}")

## General Unitary (U3)

U3(theta, phi, lambda) is the most general single-qubit gate:

$$U3(\theta, \phi, \lambda) = \begin{pmatrix} \cos(\theta/2) & -e^{i\lambda}\sin(\theta/2) \\ e^{i\phi}\sin(\theta/2) & e^{i(\phi+\lambda)}\cos(\theta/2) \end{pmatrix}$$

In [ ]:
# U3 can represent any single-qubit gate
# U3(pi, 0, pi) = X gate
circuit = qs.Circuit(1).u3(0, theta=math.pi, phi=0, lam=math.pi).measure_all()
result = qs.run(circuit, shots=100, seed=42)
print(f"U3(pi,0,pi)|0> = X|0> = {result.counts}")

# U3(pi/2, 0, pi) = H gate (up to global phase)
circuit = qs.Circuit(1).u3(0, theta=math.pi/2, phi=0, lam=math.pi).measure_all()
result = qs.run(circuit, shots=1000, seed=42)
print(f"U3(pi/2,0,pi)|0> ~ H|0> = {result.counts}")